# 小红书爆款文案之深海蓝藻保湿面膜 RAG

## 准备工作

### 依赖与环境

In [1]:
!pip freeze

annotated-doc==0.0.4
anyio @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_anyio_1782357087/work
appnope @ file:///home/conda/feedstock_root/build_artifacts/appnope_1733332318622/work
argon2-cffi @ file:///home/conda/feedstock_root/build_artifacts/argon2-cffi_1749017159514/work
argon2-cffi-bindings @ file:///Users/runner/miniforge3/conda-bld/argon2-cffi-bindings_1762509566567/work
arrow @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_arrow_1760831179/work
asttokens @ file:///home/conda/feedstock_root/build_artifacts/asttokens_1763409923949/work
async-lru @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_async-lru_1773926359/work
attrs @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_attrs_1773935801/work
babel @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_babel_1772555330/work
backports.zstd @ file:///Users/runner/miniforge3/conda-bld/bld/rattler-build_backports.zstd_1781450811/

In [2]:
!pip install "pymilvus[model]==2.5.10" "milvus-lite<2.6" "setuptools<81" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

---

In [3]:
import os

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")

In [22]:
from glob import glob

text_lines = []

for file_path in glob("rednote.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()

    text_lines += file_text.split("# ")

In [17]:
len(text_lines)

26

### 准备 LLM 和 Embedding 模型

DeepSeek 支持 OpenAI 风格的 API，您可以使用相同的 API 进行微小调整来调用 LLM。

In [6]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

定义一个 embedding 模型，使用 `milvus_model` 来生成文本嵌入。

In [7]:
from pymilvus import model as milvus_model

# OpenAI国内代理 https://api.apiyi.com/token 
embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
    model_name='text-embedding-3-large', # Specify the model name
    api_key='sk-XXX', # Provide your OpenAI API key
    base_url='https://api.apiyi.com/v1',
    dimensions=512
)

## 将数据加载到 Milvus

### 创建 Collection

In [56]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./milvus_rednote.db")

collection_name = "rednote_rag_collection"

检查 collection 是否已存在，如果存在则删除它。

In [57]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

In [63]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=512,
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)

### 插入数据

遍历文本行，创建嵌入，然后将数据插入 Milvus。

这里有一个新字段 `text`，它是在 collection schema 中未定义的字段。它将自动添加到保留的 JSON 动态字段中，该字段在高级别上可以被视为普通字段。

In [59]:
from tqdm import tqdm

data = []
# 核心清洗：过滤空字符、空白行，彻底解决 empty string 报错
text_lines = [line.strip() for line in text_lines if line.strip()]
doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    # 过滤空文本，彻底杜绝空字符串报错
    if not line or line.strip() == "":
        continue
    data.append({
        "id": i,
        "vector": doc_embeddings[i],
        "text": line.strip()
    })
    
milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|██████████| 3/3 [00:00<00:00, 8377.44it/s]


{'insert_count': 3, 'ids': [0, 1, 2], 'cost': 0}

## 构建 RAG

### 检索查询数据

In [42]:
question = "深海蓝藻保湿面膜"

In [43]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 内积距离
    output_fields=["text"],  # 返回 text 字段
)

让我们看一下查询的搜索结果

In [44]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        "\u6df1\u6d77\u84dd\u85fb\u9ad8\u6da6\u6c34\u5149\u4fdd\u6e7f\u9762\u819c\uff0c\u89c4\u683c25ml/\u7247\uff0c\u76d2\u88c510\u7247",
        0.8228024244308472
    ],
    [
        "\u6577\u819c15\u5206\u949f\u808c\u80a4\u89d2\u8d28\u5c42\u542b\u6c34\u91cf\u63d0\u534742.7%",
        0.5434918403625488
    ],
    [
        "\u4ea7\u54c1\u4e3b\u6253\u957f\u6548\u8865\u6c34\u4fee\u62a4\uff0c\u9002\u5408\u5e72\u76ae\u3001\u6df7\u5e72\u76ae\u3001\u6362\u5b63\u654f\u611f\u808c",
        0.47067776322364807
    ]
]


### 使用 LLM 获取 RAG 响应

将检索到的文档转换为字符串格式。

In [45]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [46]:
context

'深海蓝藻高润水光保湿面膜，规格25ml/片，盒装10片\n敷膜15分钟肌肤角质层含水量提升42.7%\n产品主打长效补水修护，适合干皮、混干皮、换季敏感肌'

In [47]:
question

'深海蓝藻保湿面膜'

为语言模型定义系统和用户提示。此提示是使用从 Milvus 检索到的文档组装而成的。

In [51]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。
<context>
{context}
</context>
<question>
{question}
</question>
<translated>
</translated>
"""

In [52]:
USER_PROMPT

'\n请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。\n<context>\n深海蓝藻高润水光保湿面膜，规格25ml/片，盒装10片\n敷膜15分钟肌肤角质层含水量提升42.7%\n产品主打长效补水修护，适合干皮、混干皮、换季敏感肌\n</context>\n<question>\n深海蓝藻保湿面膜\n</question>\n<translated>\n</translated>\n'

使用 DeepSeek 提供的 `deepseek-chat` 模型根据提示生成响应。

In [54]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)
rag_rednote_text = response.choices[0].message.content

深海蓝藻高润水光保湿面膜是一款规格为25ml/片、盒装10片的产品。它主打长效补水修护，敷膜15分钟可使肌肤角质层含水量提升42.7%，适合干皮、混干皮以及换季敏感肌使用。
